# Colab Minimal Run (Food101 + STL10)

Quick smoke test: download Food101, extract a small sample, train a tiny MSAE, eval on CIFAR-10 + STL-10, build tables.


In [ ]:
REPO_URL = "https://github.com/<USER>/<REPO>.git"
REPO_DIR = "sae_for_clip"

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}


In [ ]:
!pip install --upgrade pip
!pip install -r requirements.txt
!pip install -e .


In [ ]:
# Download Food101 via torchvision (dev)
from torchvision.datasets import Food101
Food101(root=f"{REPO_PATH}/data/torchvision", split="train", download=True)
FOOD_IMAGES = f"{REPO_PATH}/data/torchvision/food-101/images"


In [ ]:
# Extract activations (dev)
RUN_ACT = "colab_food101_dev_acts"
!cd {REPO_PATH} && PYTHONPATH={REPO_PATH}/src python {REPO_PATH}/scripts/extract_activations.py \
  --image_dir {FOOD_IMAGES} \
  --model ViT-B-32 \
  --pretrained openai \
  --layer visual.transformer.resblocks.0 \
  --num_samples 2000 \
  --batch_size 128 \
  --shard_size 512 \
  --out_dir {REPO_PATH}/artifacts/activation_cache \
  --run_name {RUN_ACT}


In [ ]:
# Train small MSAE
RUN_SAE = "colab_food101_dev_msae"
!PYTHONPATH=src python scripts/train_sae.py \
  --sae_type msae \
  --k_list 16,32 \
  --alpha_mode reverse \
  --input_centering dataset \
  --input_scaling dataset \
  --cache_dir artifacts/activation_cache/{RUN_ACT} \
  --dict_size 512 \
  --epochs 1 \
  --batch_size 256 \
  --lr 1e-3 \
  --l1_lambda 1e-4 \
  --device cuda \
  --run_name {RUN_SAE}


In [ ]:
# Eval CIFAR-10 + STL-10
!PYTHONPATH=src python scripts/eval_zeroshot.py --dataset cifar10 --split test --batch_size 256 --num_workers 2 --run_name c10_base
!PYTHONPATH=src python scripts/eval_zeroshot.py --dataset stl10 --split test --batch_size 256 --num_workers 2 --run_name stl10_base

!PYTHONPATH=src python scripts/eval_zeroshot.py \
  --dataset cifar10 --split test --batch_size 256 --num_workers 2 \
  --sae_checkpoint artifacts/checkpoints/{RUN_SAE}/last.pt \
  --sae_layer visual.transformer.resblocks.0 \
  --run_name c10_sae

!PYTHONPATH=src python scripts/eval_zeroshot.py \
  --dataset stl10 --split test --batch_size 256 --num_workers 2 \
  --sae_checkpoint artifacts/checkpoints/{RUN_SAE}/last.pt \
  --sae_layer visual.transformer.resblocks.0 \
  --run_name stl10_sae

!PYTHONPATH=src python scripts/make_p4_table.py \
  --eval_runs artifacts/eval/c10_sae artifacts/eval/stl10_sae \
  --checkpoint_dir artifacts/checkpoints/{RUN_SAE} \
  --out_path artifacts/eval/p4_table.md
